# Iteration 4: five-seed holdout benchmark and Optuna tuning
70/30 stratified splits; 3-fold CV inside training only. Existing ordinal codes are retained. Four selector/scaling benchmarks and tuned RFE20/standardization versions of all five model families. Local `.venv` links to the existing Experimental environment. No installation cell is required. On Colab/Kaggle, upload this project and use an environment with the listed dependencies. Google Drive mounting is optional and unchanged.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 1. Install packages directly from requirements.txt in your Drive folder
!pip install -r /content/drive/MyDrive/INFOSYS_722_Notebook/iteration4_BDAS/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 76.6 MB/s eta 0:00:00


In [3]:
!pwd

/content


In [4]:
from pathlib import Path
import os, sys

# Point directly to iteration4_BDAS inside INFOSYS_722_Notebook
PROJECT_ROOT = Path('/content/drive/MyDrive/INFOSYS_722_Notebook/iteration4_BDAS')

# Verify the script exists
assert (PROJECT_ROOT / 'holdout_workflow.py').is_file(), f'holdout_workflow.py not found at {PROJECT_ROOT}'

# Change current working directory and path
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# Print imported package versions
from importlib.metadata import version
for package in ['optuna', 'scikit-learn', 'xgboost', 'lightgbm', 'numpy', 'pandas']:
    print(package, version(package))

optuna 5.0.0
scikit-learn 1.6.1
xgboost 3.4.1
lightgbm 4.6.0
numpy 2.1.3
pandas 2.2.3


## PySpark holdout
This supplementary path runs Spark logistic regression, random forest and GBT with training-only Spark CV. It does not implement the five-family Optuna comparison above. PySpark and Java must already be available. Its preparation uses all source predictors and standardization.

In [7]:
from spark_workflow import create_spark, load_dataset, run_holdout
spark = create_spark()
frame, numeric, categorical = load_dataset(spark, PROJECT_ROOT/'data/raw/dataset_gallstone')
spark_metrics = run_holdout(spark, frame, numeric, categorical,
                            output_dir=PROJECT_ROOT/'results/spark_holdout', tune=True)

In [10]:
import json
import pandas as pd

# 1. Define path to metrics.json
json_file = PROJECT_ROOT / 'results/spark_holdout/metrics.json'

# 2. Read and display the JSON content
if json_file.exists():
    with open(json_file, 'r') as f:
        metrics_data = json.load(f)

    # Display as a formatted pandas DataFrame for clear viewing
    if isinstance(metrics_data, list):
        metrics_df = pd.DataFrame(metrics_data)
    else:
        metrics_df = pd.DataFrame([metrics_data])

    display(metrics_df)
else:
    print(f"File not found at {json_file}")

,model,rows,TN,FP,FN,TP,accuracy,roc_auc,sensitivity,specificity,precision,f1,fnr,error_rate,seed
0,gbt,96,24,24,12,36,0.625000,0.729601,0.750000,0.500000,0.600000,0.666667,0.250000,0.375000,42
1,logistic,96,30,18,2,46,0.791667,0.858941,0.958333,0.625000,0.718750,0.821429,0.041667,0.208333,42
2,random_forest,96,31,17,18,30,0.635417,0.710937,0.625000,0.645833,0.638298,0.631579,0.375000,0.364583,42
3,gbt,96,33,15,8,40,0.760417,0.834635,0.833333,0.687500,0.727273,0.776699,0.166667,0.239583,7
4,logistic,96,34,14,5,43,0.802083,0.850260,0.895833,0.708333,0.754386,0.819048,0.104167,0.197917,7
5,random_forest,96,35,13,12,36,0.739583,0.835938,0.750000,0.729167,0.734694,0.742268,0.250000,0.260417,7
6,gbt,96,35,13,11,37,0.750000,0.814670,0.770833,0.729167,0.740000,0.755102,0.229167,0.250000,21
7,logistic,96,33,15,6,42,0.781250,0.855035,0.875000,0.687500,0.736842,0.800000,0.125000,0.218750,21
8,random_forest,96,35,13,10,38,0.760417,0.821181,0.791667,0.729167,0.745098,0.767677,0.208333,0.239583,21
9,gbt,96,34,14,13,35,0.718750,0.798611,0.729167,0.708333,0.714286,0.721649,0.270833,0.281250,84
